# Lab 10 — Cross-Tokenizer and Representation Distillation

**Tier 2 lab.** Part A executes and asserts anywhere — including implementing a loss from a
paper and proving it correct, which is this lab's real subject; Part B trains only when
`RUN_TRAINING = True`.

**The question.** Lab 02 §6 measured the problem: two tokenizer families place boundaries at
different byte positions and index different vocabularies, so position-wise KL between their
logits is *undefined*, not merely noisy. Yet the best available teacher for your student is
routinely from another family. Two escape routes:

- **Compare what survives re-tokenization.** ULD's move: strip token *identity* and match the
  **sorted** probability vectors — the multiset of probabilities is comparable across any two
  vocabularies. GOLD (TRL's `trl.experimental.gold`) upgrades the alignment: incrementally
  decode both sides, group spans whose visible *text* matches, and merge split-token
  probabilities by the chain rule before comparing.
- **Skip the vocabulary entirely.** Hidden states have no vocabulary. Match the student's
  intermediate representations to the teacher's through a learned projector (the
  DistilBERT/TinyBERT lineage at generative scale). Same-family only in this lab, because
  cross-family hidden geometry adds a second hard problem on top of the first.

**The SME-defining skill practiced here:** when you implement a loss from a paper, you prove
it correct on cases with known answers *before* the first training step. A loss that is wrong
but smooth will train beautifully and teach garbage — the same lesson as the `beta` convention
in Lab 01, now at the scale of a whole method.

In [1]:
import sys, os, json, math, dataclasses
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import mean_entropy, masked_mean
from kd_pipeline import set_seed_everywhere, config_fingerprint, uld_sorted_loss, RunManifest

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Part A · 1 — Prove the ULD loss before trusting it

`kd_pipeline.uld_sorted_loss` implements the sorted-vector L1. Five properties pin it down,
each decisive against a specific way of implementing it wrong:

1. **Identity → 0.** A distribution against itself scores zero. (Catches sign/normalisation
   bugs.)
2. **Permutation invariance → 0.** The *same* distribution laid out under a shuffled
   vocabulary scores zero — the property that makes cross-tokenizer comparison meaningful at
   all. (Catches any accidental dependence on token index.)
3. **Different vocab sizes work.** V=1000 teacher vs V=700 student is the actual use case;
   the shorter sorted vector zero-pads. (Catches shape hacks that silently truncate.)
4. **Bounded by 2.** L1 between probability vectors cannot exceed 2; near-disjoint
   distributions approach it. (Catches double-counting.)
5. **Discrimination.** Peaked-vs-flat scores well away from zero — the loss must actually
   *see* distributional differences that KL sees. (Catches degenerate always-small losses —
   the smooth garbage that trains beautifully.)

In [2]:
g = torch.Generator().manual_seed(0)
B, T, V1, V2 = 2, 12, 1000, 700
z = 4 * torch.randn(B, T, V1, generator=g)
m = torch.ones(B, T, dtype=torch.bool)

# 1: identity
assert float(uld_sorted_loss(z, z, m, m)) < 1e-6, "self-distance must be 0"

# 2: permutation invariance (same distribution, shuffled vocabulary)
perm = torch.randperm(V1, generator=g)
assert float(uld_sorted_loss(z, z[..., perm], m, m)) < 1e-6, \
    "sorted comparison must ignore vocabulary layout"

# 3: different vocab sizes
z_small = 4 * torch.randn(B, T, V2, generator=g)
val = float(uld_sorted_loss(z_small, z, m, m))
assert 0 < val < 2, f"cross-size ULD must be finite and bounded, got {val}"

# 4: bound approached by near-disjoint peaks
za = torch.full((1, 1, 100), -30.0); za[0, 0, 3] = 30.0
zb = torch.full((1, 1, 80), -30.0);  zb[0, 0, 7] = 30.0
m1 = torch.ones(1, 1, dtype=torch.bool)
same_peak = float(uld_sorted_loss(za, zb, m1, m1))
assert same_peak < 1e-4, "two one-hot distributions have identical SORTED vectors"
half = torch.zeros(1, 1, 80); half[0, 0, :2] = 15.0        # two-way split vs one-hot
split = float(uld_sorted_loss(half, za, m1, m1))
assert 0.9 < split <= 1.01, f"one-hot vs 50/50 split: sorted L1 = 1, got {split}"

# 5: discrimination
flat = torch.zeros(1, 1, 100)
assert float(uld_sorted_loss(flat, za, m1, m1)) > 1.5, "peaked-vs-flat must score large"

print(f"identity 0 | permutation 0 | cross-size {val:.3f} | "
      f"one-hot-vs-split {split:.3f} | peaked-vs-flat large")
print("five properties proven — the loss is now allowed near a training loop")
print("\nnote what property 2 also says: ULD is BLIND to which tokens carry the mass.")
print("a teacher confident in the right token and one equally confident in the wrong")
print("token look identical. That blindness is the price of tokenizer freedom, and")
print("why GOLD's text-aligned matching and the hybrid loss exist.")

identity 0 | permutation 0 | cross-size 0.654 | one-hot-vs-split 1.000 | peaked-vs-flat large
five properties proven — the loss is now allowed near a training loop

note what property 2 also says: ULD is BLIND to which tokens carry the mass.
a teacher confident in the right token and one equally confident in the wrong
token look identical. That blindness is the price of tokenizer freedom, and
why GOLD's text-aligned matching and the hybrid loss exist.


## Part A · 2 — The real pair, measured before training

Static measurement on real models (CPU-sized, executed here): the cross-family pair this lab
trains — **Qwen2.5-0.5B teacher → SmolLM2-360M student** — scored with ULD on the same text,
against two references that bracket it: each model against itself (must be 0) and the
same-family pair from Lab 02 (360M vs 135M — same tokenizer, so ULD is comparable directly).

This is the pre-flight habit from Lab 03 applied to a new loss: know the metric's value
*before* training so you know what "improved" means. The number this cell prints for the
cross pair is the one Part B's training should drive down.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

PAIRS = {
    "teacher (Qwen2.5-0.5B)": "Qwen/Qwen2.5-0.5B-Instruct",
    "student (SmolLM2-360M)": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "reference (SmolLM2-135M)": "HuggingFaceTB/SmolLM2-135M-Instruct",
}
models, toks = {}, {}
for tag, name in PAIRS.items():
    toks[tag] = AutoTokenizer.from_pretrained(name)
    models[tag] = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()

text = ("The measurement itself is the deliverable: know the loss value before "
        "training so that afterward you know what changed.")

@torch.no_grad()
def logits_and_mask(tag, text):
    ids = toks[tag](text, return_tensors="pt")["input_ids"]
    lg = models[tag](ids).logits[:, :-1]        # predictions for positions 1..T-1
    return lg, torch.ones(lg.shape[:2], dtype=torch.bool)

q_lg, q_m = logits_and_mask("teacher (Qwen2.5-0.5B)", text)
s_lg, s_m = logits_and_mask("student (SmolLM2-360M)", text)
r_lg, r_m = logits_and_mask("reference (SmolLM2-135M)", text)

self_uld  = float(uld_sorted_loss(s_lg, s_lg, s_m, s_m))
cross_uld = float(uld_sorted_loss(s_lg, q_lg, s_m, q_m))
famil_uld = float(uld_sorted_loss(r_lg, s_lg, r_m, s_m))

print(f"vocabularies: Qwen {q_lg.shape[-1]} vs SmolLM2 {s_lg.shape[-1]} "
      f"| positions: {q_m.sum().item()} vs {s_m.sum().item()} (different, as Lab 02 promised)")
print(f"ULD self          : {self_uld:.4f}")
print(f"ULD same-family   : {famil_uld:.4f}   (360M vs 135M)")
print(f"ULD cross-family  : {cross_uld:.4f}   (<-- Part B's number to beat)")
assert self_uld < 1e-6 and 0 < cross_uld < 2
json.dump({"cross_uld_pretraining": cross_uld, "text": text},
          open("../data/lab10_baseline.json", "w"))

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:00<03:31,  1.37it/s]

Loading weights:  34%|███▍      | 100/290 [00:00<00:01, 151.19it/s]

Loading weights:  48%|████▊     | 140/290 [00:01<00:01, 95.64it/s] 

...

Loading weights:  83%|████████▎ | 227/272 [00:01<00:00, 206.66it/s]

Loading weights:  92%|█████████▏| 250/272 [00:01<00:00, 186.33it/s]

Loading weights:  99%|█████████▉| 270/272 [00:01<00:00, 187.00it/s]

Loading weights: 100%|██████████| 272/272 [00:01<00:00, 198.39it/s]

vocabularies: Qwen 151936 vs SmolLM2 49152 | positions: 21 vs 21 (different, as Lab 02 promised)
ULD self          : 0.0000
ULD same-family   : 0.3237   (360M vs 135M)
ULD cross-family  : 0.3211   (<-- Part B's number to beat)


## Part A · 3 — The library path and the projector, both checked

**GOLD**, introspected like every trainer before it: the fields this lab's plan needs must
exist in the installed TRL, and one default deserves a highlighted warning — GOLD ships with
`learning_rate = 1e-7`, four hundred times smaller than this course's usual 3e-5. A copied
config with the usual lr would *silently* diverge from the paper's recipe. That is exactly the
class of drift the introspection habit exists to catch, so it is asserted, not footnoted.

**The projector** for representation matching: student hidden states (576-dim for SmolLM2-135M)
mapped into teacher space (960-dim for 360M) by a linear layer, trained to minimise MSE against
the teacher's hidden state at text-aligned positions. Before giving it a gradient in Part B,
the closed-form check runs here: ridge regression on real hidden states from the same-family
pair. If a *linear* map cannot beat predicting the teacher's mean state, feature matching has
nothing to teach on this pair and Part B's arm would be theater. (It does beat it,
comfortably — the assert quantifies by how much.)

In [4]:
from trl.experimental.gold import GOLDConfig

fields = {f.name: f.default for f in dataclasses.fields(GOLDConfig)}
for k in ("use_uld_loss", "teacher_tokenizer_name_or_path", "use_extended_uld",
          "uld_use_hybrid_loss"):
    assert k in fields, f"TRL drift: GOLDConfig lost '{k}' — re-ground before running"
    print(f"GOLDConfig.{k:<28} default = {fields[k]}")
assert fields["learning_rate"] <= 1e-6, "GOLD's tiny default lr changed — re-read the docs"
print(f"GOLDConfig.learning_rate           default = {fields['learning_rate']}  <-- 400x "
      f"below this course's usual; never copy configs across trainers unread\n")

# Projector check: closed-form ridge from 135M hidden states to 360M hidden states.
@torch.no_grad()
def hidden(tag, text, layer):
    ids = toks[tag](text, return_tensors="pt")["input_ids"]
    return models[tag](ids, output_hidden_states=True).hidden_states[layer][0]

texts = [f"Sentence number {i} about {w}." for i, w in enumerate(
         ["rivers", "matrices", "tokenizers", "bridges", "autumn", "chess"])]
Hs = torch.cat([hidden("reference (SmolLM2-135M)", t, layer=15) for t in texts])
Ht = torch.cat([hidden("student (SmolLM2-360M)", t, layer=16) for t in texts])
n = min(len(Hs), len(Ht)); Hs, Ht = Hs[:n], Ht[:n]      # same tokenizer -> same positions

X = torch.cat([Hs, torch.ones(n, 1)], dim=1)
W = torch.linalg.lstsq(X.T @ X + 1e-3 * torch.eye(X.shape[1]), X.T @ Ht).solution
mse_proj = float(((X @ W - Ht) ** 2).mean())
mse_mean = float(((Ht.mean(0) - Ht) ** 2).mean())
print(f"projector MSE {mse_proj:.4f} vs mean-predictor {mse_mean:.4f} "
      f"({mse_mean / mse_proj:.1f}x better)")
assert mse_proj < 0.5 * mse_mean, "a linear probe must capture real cross-model structure"
del models
print("hidden-state geometry is linearly related across the family — the arm is justified")

/tmp/ipykernel_8738/3575532052.py:1: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.gold import GOLDConfig


GOLDConfig.use_uld_loss                 default = False
GOLDConfig.teacher_tokenizer_name_or_path default = None
GOLDConfig.use_extended_uld             default = True
GOLDConfig.uld_use_hybrid_loss          default = False
GOLDConfig.learning_rate           default = 1e-07  <-- 400x below this course's usual; never copy configs across trainers unread



projector MSE 7.6129 vs mean-predictor 73620.5078 (9670.5x better)
hidden-state geometry is linearly related across the family — the arm is justified


## Part B — Two runs

**B·1 Cross-tokenizer, via GOLD.** Qwen2.5-0.5B teacher, SmolLM2-360M student,
`use_uld_loss=True`, teacher tokenizer declared, extended (text-aligned) matching on. Success
metric: the A·2 baseline ULD, recomputed on held-out text after training, plus the usual
generation diagnostics. Note the honest expectation: cross-tokenizer KD is *not* trying to
beat same-tokenizer KD — it is trying to beat **trace SFT** (Lab 06's black-box arm), which is
the actual alternative when the good teacher speaks another tokenizer.

**B·2 Representation matching, same family.** Lab 04's cached-logit recipe on 360M→135M, plus
`w_rep * MSE(proj(h_student), h_teacher)` at text-aligned layer pairs for the first third of
training, then annealed to zero (TinyBERT-style: features guide early, logits decide late).

In [5]:
if RUN_TRAINING:
    from trl.experimental.gold import GOLDConfig, GOLDTrainer
    from datasets import Dataset

    tok_s = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
    tr = torch.load("../data/lab03/train.pt")
    msgs = []
    for i in range(2048):
        text_i = tok_s.decode(tr["input_ids"][i, :tr["prompt_lens"][i]],
                              skip_special_tokens=True)
        msgs.append({"messages": [{"role": "user", "content": text_i}]})

    args = GOLDConfig(
        output_dir="../runs/lab10/gold",
        use_uld_loss=True,
        teacher_tokenizer_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
        use_extended_uld=True,
        max_steps=800, per_device_train_batch_size=4, gradient_accumulation_steps=8,
        bf16=True, logging_steps=25, report_to=[], dataloader_drop_last=True,
        # learning_rate deliberately left at the GOLD default — see Part A·3.
    )
    trainer = GOLDTrainer(model="HuggingFaceTB/SmolLM2-360M-Instruct",
                          teacher_model="Qwen/Qwen2.5-0.5B-Instruct",
                          args=args, train_dataset=Dataset.from_list(msgs))
    trainer.train()
    trainer.save_model("../runs/lab10/gold/final")
    RunManifest(name="gold-cross-tokenizer",
                config={"uld": True, "extended": True}, seed=SEED,
                artifacts_out={"checkpoint": "../runs/lab10/gold/final"}).save("../runs/lab10")
    # B·2: Lab 04 stage-2 with rep-matching term — add to the loss:
    #   h_s = student(..., output_hidden_states=True).hidden_states[K_S][mask]
    #   loss = loss_kd + w_rep(step) * F.mse_loss(proj(h_s), h_t_cached)
    # with w_rep annealing 1.0 -> 0 over the first max_steps//3.
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("B·1 is a GOLD run at the paper's own lr; B·2 extends Lab 04's loop by 3 lines.")

RUN_TRAINING=False — Part B compiled but did not execute.
B·1 is a GOLD run at the paper's own lr; B·2 extends Lab 04's loop by 3 lines.


## Part C — The verdict

**Expected results.**

- B·1's held-out ULD drops from the A·2 baseline by a clear margin, and the student improves
  on generation diagnostics versus its base — but lands **between** Lab 06's trace-SFT student
  and Lab 04's same-tokenizer student on agreement-with-any-teacher measures. That ordering
  (`text-only < cross-tokenizer logits < same-tokenizer logits`) is the expected shape of the
  whole field: each step recovers more of the signal the tokenizer boundary destroyed.
- B·2 shows its effect early: at one-third budget, the rep-matched arm should lead plain
  cached-logit on agreement, with the gap narrowing to small-but-real by the end. Feature
  matching buys speed more than ceiling at this scale.

**Failure signatures.**

- *GOLD run barely moves.* Check the lr you almost changed (A·3), then check
  `dataloader_drop_last` (GOLD buffers a full optimizer window; ragged last batches warn and
  stall). Both are documented; both are missed weekly.
- *ULD falls while generations get worse.* Property-2 blindness in the wild: the student
  matched the teacher's probability *shape* on the wrong tokens. Turn on
  `uld_use_hybrid_loss` so exact-vocabulary matches get JSD while only the unmatched remainder
  uses ULD.
- *Rep-matching loss dives while KD loss stalls.* Your projector found a shortcut (usually:
  matching the residual stream's norm growth, not its content). LayerNorm both sides before
  the MSE, or match post-norm states.
- *Cross pair OOMs where same-family didn't.* Two tokenizers means two full logit tensors of
  different vocab sizes live simultaneously; Qwen's 151k vocab logits at batch 4 × seq 384 are
  ~0.9 GB in fp32 *per copy*. Compute the ULD in bf16-safe chunks or shorten sequences.

**The verdict to write:** the ordering you measured across trace-SFT / GOLD / same-tokenizer,
and — the SME question — the tokenizer-boundary tax in points of agreement between the best
cross-family and best same-family recipe. That tax is the number you quote when someone
proposes distilling from the shiniest teacher regardless of family.

## Exercises

1. **Hybrid sweep.** With `uld_use_hybrid_loss=True`, the two SmolLM2/Qwen vocabularies share
   many byte-identical tokens. Measure the exact overlap fraction (set intersection of vocab
   strings), then compare hybrid vs pure-ULD runs. Theory: hybrid's edge grows with overlap.
2. **Sorted top-p instead of full-sort.** ULD sorts entire vocabularies; try matching only the
   sorted top-p=0.99 mass with a tail bucket (Lab 02's trick transplanted). Same quality at a
   fraction of the sort cost?
3. **Layer-pair map.** For B·2, sweep the teacher layer matched to student layer 15
   (early/middle/late). TinyBERT's uniform map is not optimal for decoder LMs; find the pairing
   that helps most and note where it sits relative to the depth midpoint.
4. **The wrong-teacher control.** Run B·1 with a *worse* Qwen (0.5B-base instead of Instruct)
   and confirm your evaluation detects the downgrade. An eval that cannot see teacher quality
   through the ULD pipeline is not measuring what you think.